# 💧 Prédiction de la Potabilité de l'Eau
## Analyse Exploratoire & Pipeline de Modélisation

---

**Projet** : AQUA SENS  
**Dataset** : [Water Potability — Kaggle](https://www.kaggle.com/datasets/adityakadiwal/water-potability)  
**Auteur** : PLBD  
**Date** : 2026

---

### Contexte et contraintes du projet

L'accès à une eau potable de qualité est un enjeu de santé publique majeur.  
Ce projet vise à construire un modèle de classification binaire capable de prédire  
si une eau est **non potable (1)** ou **potable (0)** à partir de mesures physico-chimiques.

#### ⚠️ Contraintes opérationnelles importantes

Le dataset contient **9 variables physico-chimiques**. Cependant, notre modèle  
n'en utilise que **4** : `ph`, `Solids`, `Conductivity`, `Turbidity`.

Ce choix est contraint par des **considérations économiques** :

| Variable | Capteur requis | Coût estimé | Statut |
|---|---|---|---|
| ph | Électrode pH | 💲 Faible | ✅ Retenu |
| Solids (TDS) | Conductimètre TDS | 💲 Faible | ✅ Retenu |
| Conductivity | Conductimètre EC | 💲 Faible | ✅ Retenu |
| Turbidity | Turbidimètre | 💲 Faible | ✅ Retenu |
| Hardness | Titrimètre + réactifs | 💲💲 Moyen | ❌ Écarté |
| Chloramines | Spectrophotomètre | 💲💲💲 Élevé | ❌ Écarté |
| Sulfate | Chromatographe ionique | 💲💲💲 Élevé | ❌ Écarté |
| Organic_carbon | Analyseur TOC | 💲💲💲💲 Très élevé | ❌ Écarté |
| Trihalomethanes | GC-MS | 💲💲💲💲 Très élevé | ❌ Écarté |

> **Limite fondamentale** : Comme nous le verrons dans l'analyse de corrélation,  
> les 4 variables retenues ont une **faible corrélation individuelle avec la cible**.  
> Les variables les plus prédictives (Sulfate, Organic_carbon, Trihalomethanes)  
> sont précisément celles que le budget ne permet pas d'acquérir.  
> Ce compromis coût/performance est documenté et assumé dans ce projet.

---

### Structure du notebook

1. Configuration & imports  
2. Chargement des données brutes  
3. Vue d'ensemble du dataset complet (9 variables)  
4. Analyse des valeurs manquantes  
5. Distribution des variables & détection des outliers  
6. **Analyse de corrélation complète** (toutes variables × cible)  
7. Analyse bivariée par classe  
8. Contrainte de sélection des features — justification quantifiée  
9. Pipeline de prétraitement  
10. Rééquilibrage des classes (SMOTETomek)  
11. Aperçu des données prêtes pour la modélisation  

**Analyses de clustering (Section 13)**  
13.1 KMeans(k=2) → ARI score — structure naturelle des données  
13.2 GMM comme feature engineering — gain de performance  
13.3 Isolation Forest — détection d'anomalies standalone  
13.4 Synthèse & recommandations  


In [54]:
# ── Imports ─────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import sys
import os
from pathlib import Path

# ── Résolution robuste du chemin vers src/ ────────────────────────────────
# Cherche le dossier src/ depuis le répertoire du notebook, puis remonte
# jusqu'à 3 niveaux si nécessaire (couvre : notebooks/, racine, sous-dossiers)
def _find_src() -> Path:
    candidates = [Path.cwd()]
    for _ in range(3):
        candidates.append(candidates[-1].parent)
    for base in candidates:
        for sub in [base / "src", base]:
            if (sub / "config.py").exists():
                return sub
    raise FileNotFoundError(
        "config.py introuvable. Vérifiez que le notebook est lancé "
        "depuis la racine du projet ou que src/ est dans le bon répertoire."
    )

SRC_PATH = _find_src()
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"✓ src/ résolu : {SRC_PATH}")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import pointbiserialr, chi2_contingency

# Config projet
from config import FEATURES, TARGET, CLASS_LABELS, RANDOM_STATE

# Chemin des données — résolution automatique depuis la racine projet
PROJECT_ROOT = SRC_PATH.parent if SRC_PATH.name == "src" else SRC_PATH
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "water_potability.csv"
OUTPUTS_PATH  = PROJECT_ROOT / "outputs" / "figures"
OUTPUTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Données         : {RAW_DATA_PATH}")
print(f"✓ Figures outputs : {OUTPUTS_PATH}")

# Style global
plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         11,
})
PALETTE = {"Non potable": "#E84855", "Potable": "#2E86AB"}
PALETTE_BIN = ["#2E86AB", "#E84855"]

print(f"\n✓ Imports OK")
print(f"  Features retenues : {FEATURES}")
print(f"  Variable cible    : {TARGET}")
print(f"  Classes           : {CLASS_LABELS}")


✓ src/ résolu : c:\Users\HP\PLBD\src
✓ Données         : c:\Users\HP\PLBD\data\raw\water_potability.csv
✓ Figures outputs : c:\Users\HP\PLBD\outputs\figures

✓ Imports OK
  Features retenues : ['ph', 'Solids', 'Conductivity', 'Turbidity']
  Variable cible    : Potability
  Classes           : {0: 'Potable', 1: 'Non potable'}


## 2. Chargement des données brutes

On charge le CSV **brut** (avant tout prétraitement) pour analyser l'ensemble des 9 variables.

In [55]:
# Chargement du CSV brut — toutes les 9 variables
RAW_PATH = str(RAW_DATA_PATH)

df_raw = pd.read_csv(RAW_PATH)

# Relabélisation immédiate pour cohérence : 1 = Non potable, 0 = Potable
df_raw[TARGET] = 1 - df_raw[TARGET]

# Toutes les variables physico-chimiques
ALL_FEATURES = [c for c in df_raw.columns if c != TARGET]
EXCLUDED_FEATURES = [f for f in ALL_FEATURES if f not in FEATURES]

print(f"Shape : {df_raw.shape[0]} observations × {df_raw.shape[1]} variables")
print(f"\nVariables disponibles  ({len(ALL_FEATURES)}) : {ALL_FEATURES}")
print(f"Variables retenues     ({len(FEATURES)})  : {FEATURES}")
print(f"Variables écartées     ({len(EXCLUDED_FEATURES)})  : {EXCLUDED_FEATURES}")
df_raw.head()


Shape : 3276 observations × 10 variables

Variables disponibles  (9) : ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']
Variables retenues     (4)  : ['ph', 'Solids', 'Conductivity', 'Turbidity']
Variables écartées     (5)  : ['Hardness', 'Chloramines', 'Sulfate', 'Organic_carbon', 'Trihalomethanes']


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,1
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,1
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,1
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,1
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,1


## 3. Vue d'ensemble du dataset complet

In [56]:
# Statistiques descriptives complètes
stats_df = df_raw.describe().T
stats_df["missing"]     = df_raw.isnull().sum()
stats_df["missing_pct"] = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
stats_df["skewness"]    = df_raw.skew()
stats_df["kurtosis"]    = df_raw.kurt()

# Marquer les variables retenues vs écartées
stats_df["statut"] = stats_df.index.map(
    lambda c: "✅ Retenu" if c in FEATURES else ("🎯 Cible" if c == TARGET else "❌ Écarté (coût)")
)

display(stats_df[["statut","count","mean","std","min","25%","50%","75%","max",
                   "missing","missing_pct","skewness","kurtosis"]].round(3))


,statut,count,mean,std,min,25%,50%,75%,max,missing,missing_pct,skewness,kurtosis
ph,✅ Retenu,2785.0,7.081,1.594,0.000,6.093,7.037,8.062,14.000,491,14.99,0.026,0.720
Hardness,❌ Écarté (coût),3276.0,196.369,32.880,47.432,176.851,196.968,216.667,323.124,0,0.00,-0.039,0.616
Solids,✅ Retenu,3276.0,22014.093,8768.571,320.943,15666.690,20927.834,27332.762,61227.196,0,0.00,0.622,0.443
Chloramines,❌ Écarté (coût),3276.0,7.122,1.583,0.352,6.127,7.130,8.115,13.127,0,0.00,-0.012,0.590
Sulfate,❌ Écarté (coût),2495.0,333.776,41.417,129.000,307.699,333.074,359.950,481.031,781,23.84,-0.036,0.648
Conductivity,✅ Retenu,3276.0,426.205,80.824,181.484,365.734,421.885,481.792,753.343,0,0.00,0.264,-0.277
Organic_carbon,❌ Écarté (coût),3276.0,14.285,3.308,2.200,12.066,14.218,16.558,28.300,0,0.00,0.026,0.044
Trihalomethanes,❌ Écarté (coût),3114.0,66.396,16.175,0.738,55.845,66.622,77.337,124.000,162,4.95,-0.083,0.239
Turbidity,✅ Retenu,3276.0,3.967,0.780,1.450,3.440,3.955,4.500,6.739,0,0.00,-0.008,-0.063
Potability,🎯 Cible,3276.0,0.610,0.488,0.000,0.000,1.000,1.000,1.000,0,0.00,-0.451,-1.798


## 4. Analyse des valeurs manquantes

In [57]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot % manquants
missing = df_raw.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

colors = ["#E84855" if c not in FEATURES else "#2E86AB" for c in missing.index]
bars = axes[0].barh(missing.index, missing.values, color=colors, edgecolor="white")
axes[0].set_xlabel("% valeurs manquantes")
axes[0].set_title("Taux de valeurs manquantes par variable", fontweight="bold")
axes[0].axvline(5, color="gray", linestyle="--", linewidth=1, label="Seuil 5%")
axes[0].legend()
for bar, val in zip(bars, missing.values):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{val:.1f}%", va="center", fontsize=9)

# Heatmap des NaN
sns.heatmap(df_raw.isnull(), yticklabels=False, cbar=False,
            cmap=["#f0f0f0", "#E84855"], ax=axes[1])
axes[1].set_title("Carte des valeurs manquantes\n(rouge = NaN)", fontweight="bold")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha="right")

# Légende couleurs
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#2E86AB", label="Variable retenue"),
                   Patch(facecolor="#E84855", label="Variable écartée (coût)")]
axes[0].legend(handles=legend_elements, loc="lower right", fontsize=9)

plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "missing_values.png"), dpi=120, bbox_inches="tight")
plt.show()
print(f"\nVariables avec NaN : {df_raw.isnull().any().sum()}")
print(missing.to_frame("% manquant"))



Variables avec NaN : 3
                 % manquant
Sulfate           23.840049
ph                14.987790
Trihalomethanes    4.945055


## 5. Distribution des variables & détection des outliers

On analyse la distribution de **toutes les variables** en distinguant les deux classes.  
La couleur identifie le statut de chaque variable dans notre projet.


In [58]:
n_cols = 3
n_rows = int(np.ceil(len(ALL_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, feat in enumerate(ALL_FEATURES):
    ax = axes[i]
    color_0 = PALETTE_BIN[0]
    color_1 = PALETTE_BIN[1]

    for label_val, label_name, color in [
        (0, "Potable",      color_0),
        (1, "Non potable",  color_1),
    ]:
        data = df_raw[df_raw[TARGET] == label_val][feat].dropna()
        ax.hist(data, bins=40, alpha=0.55, color=color, label=label_name, density=True)
        # KDE
        from scipy.stats import gaussian_kde
        if len(data) > 10:
            kde = gaussian_kde(data)
            x = np.linspace(data.min(), data.max(), 200)
            ax.plot(x, kde(x), color=color, linewidth=2)

    status = "✅" if feat in FEATURES else "❌"
    ax.set_title(f"{status} {feat}", fontweight="bold",
                 color="#2E86AB" if feat in FEATURES else "#888888")
    ax.set_xlabel("")
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

# Masquer les axes vides
for j in range(len(ALL_FEATURES), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribution des variables par classe\n"
             "✅ = Retenu  |  ❌ = Écarté (coût capteur)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "distributions_by_class.png"), dpi=120, bbox_inches="tight")
plt.show()


In [59]:
# Détection des outliers — méthode IQR (factor=1.5)
print("=" * 60)
print("  RAPPORT OUTLIERS IQR (factor=1.5)")
print("=" * 60)
records = []
for feat in ALL_FEATURES:
    s = df_raw[feat].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lower) | (s > upper)).sum()
    records.append({
        "Variable": feat,
        "Statut":   "✅ Retenu" if feat in FEATURES else "❌ Écarté",
        "Q1": round(q1, 2), "Q3": round(q3, 2), "IQR": round(iqr, 2),
        "Borne inf.": round(lower, 2), "Borne sup.": round(upper, 2),
        "N outliers": n_out,
        "% outliers": round(n_out / len(s) * 100, 2),
    })

outlier_df = pd.DataFrame(records).sort_values("% outliers", ascending=False)
display(outlier_df)


  RAPPORT OUTLIERS IQR (factor=1.5)


,Variable,Statut,Q1,Q3,IQR,Borne inf.,Borne sup.,N outliers,% outliers
1,Hardness,❌ Écarté,176.85,216.67,39.82,117.13,276.39,83,2.53
3,Chloramines,❌ Écarté,6.13,8.11,1.99,3.15,11.10,61,1.86
0,ph,✅ Retenu,6.09,8.06,1.97,3.14,11.02,46,1.65
4,Sulfate,❌ Écarté,307.70,359.95,52.25,229.32,438.33,41,1.64
2,Solids,✅ Retenu,15666.69,27332.76,11666.07,-1832.42,44831.87,47,1.43
7,Trihalomethanes,❌ Écarté,55.84,77.34,21.49,23.61,109.58,33,1.06
6,Organic_carbon,❌ Écarté,12.07,16.56,4.49,5.33,23.30,25,0.76
8,Turbidity,✅ Retenu,3.44,4.50,1.06,1.85,6.09,19,0.58
5,Conductivity,✅ Retenu,365.73,481.79,116.06,191.65,655.88,11,0.34


## 6. Analyse de corrélation complète

### 6.1 Corrélation entre toutes les variables (matrice)


In [60]:
# Matrice de corrélation — dataset complet (sans NaN pour la corrélation)
corr_matrix = df_raw.corr(method="pearson")

# Masque triangle supérieur
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))
cmap = sns.diverging_palette(220, 20, as_cmap=True)

sns.heatmap(
    corr_matrix, mask=mask, cmap=cmap, center=0,
    annot=True, fmt=".2f", annot_kws={"size": 9},
    linewidths=0.5, ax=ax,
    vmin=-1, vmax=1,
    cbar_kws={"shrink": 0.8, "label": "Corrélation de Pearson"},
)

ax.set_title("Matrice de corrélation — Toutes les variables\n"
             "(dataset complet, 9 features + cible)",
             fontsize=13, fontweight="bold", pad=15)

# Encadrer les variables retenues
retained_idx = [list(corr_matrix.columns).index(f) for f in FEATURES if f in corr_matrix.columns]
for idx in retained_idx:
    ax.add_patch(plt.Rectangle((idx, idx), 1, 1, fill=False,
                                edgecolor="#2E86AB", lw=2.5))

plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "correlation_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()


### 6.2 Corrélation de chaque variable avec la cible `Potability`

**Méthode** : Point-Biserial (variable continue × cible binaire) — plus adaptée que Pearson pour une cible binaire.

In [61]:
# Corrélation point-bisériale avec la cible
from scipy.stats import pointbiserialr

pb_records = []
for feat in ALL_FEATURES:
    sub = df_raw[[feat, TARGET]].dropna()
    corr, pval = pointbiserialr(sub[TARGET], sub[feat])
    pb_records.append({
        "Variable":      feat,
        "Statut":        "✅ Retenu" if feat in FEATURES else "❌ Écarté (coût)",
        "Corrélation":   round(corr, 4),
        "|Corrélation|": round(abs(corr), 4),
        "p-value":       round(pval, 4),
        "Significatif":  "✓" if pval < 0.05 else "✗",
    })

pb_df = (pd.DataFrame(pb_records)
           .sort_values("|Corrélation|", ascending=False)
           .reset_index(drop=True))
pb_df.index += 1

display(pb_df)

print("\n⚠️  CONSTAT CLÉ :")
print(f"  → Corrélation max (toutes variables)  : "
      f"{pb_df['|Corrélation|'].max():.4f}  ({pb_df.iloc[0]['Variable']})")
print(f"  → Corrélation max (variables retenues) : "
      f"{pb_df[pb_df['Statut'].str.contains('Retenu')]['|Corrélation|'].max():.4f}")
print(f"  → Corrélation min (variables retenues) : "
      f"{pb_df[pb_df['Statut'].str.contains('Retenu')]['|Corrélation|'].min():.4f}")


,Variable,Statut,Corrélation,|Corrélation|,p-value,Significatif
1,Solids,✅ Retenu,-0.0337,0.0337,0.0535,✗
2,Organic_carbon,❌ Écarté (coût),0.0300,0.0300,0.0860,✗
3,Chloramines,❌ Écarté (coût),-0.0238,0.0238,0.1736,✗
4,Sulfate,❌ Écarté (coût),0.0236,0.0236,0.2391,✗
5,Hardness,❌ Écarté (coût),0.0138,0.0138,0.4285,✗
6,Conductivity,✅ Retenu,0.0081,0.0081,0.6419,✗
7,Trihalomethanes,❌ Écarté (coût),-0.0071,0.0071,0.6908,✗
8,ph,✅ Retenu,0.0036,0.0036,0.8512,✗
9,Turbidity,✅ Retenu,-0.0016,0.0016,0.9279,✗



⚠️  CONSTAT CLÉ :
  → Corrélation max (toutes variables)  : 0.0337  (Solids)
  → Corrélation max (variables retenues) : 0.0337
  → Corrélation min (variables retenues) : 0.0016


In [62]:
# Visualisation comparative : variables retenues vs écartées
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors_bar = ["#2E86AB" if "Retenu" in s else "#E84855" for s in pb_df["Statut"]]
bars = axes[0].barh(pb_df["Variable"][::-1], pb_df["Corrélation"][::-1],
                    color=colors_bar[::-1], edgecolor="white")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].axvline(0.1,  color="gray", linestyle=":", linewidth=1, alpha=0.7)
axes[0].axvline(-0.1, color="gray", linestyle=":", linewidth=1, alpha=0.7)
axes[0].set_xlabel("Corrélation Point-Bisériale avec Potabilité")
axes[0].set_title("Corrélation de chaque variable avec la cible\n"
                   "(valeur signée)", fontweight="bold")

from matplotlib.patches import Patch
legend_el = [Patch(facecolor="#2E86AB", label="Variable retenue (budget)"),
             Patch(facecolor="#E84855", label="Variable écartée (coût capteur)")]
axes[0].legend(handles=legend_el, fontsize=9, loc="lower right")

# Valeurs absolues — plus lisible pour comparer les forces
bars2 = axes[1].barh(pb_df["Variable"][::-1], pb_df["|Corrélation|"][::-1],
                      color=colors_bar[::-1], edgecolor="white")
axes[1].axvline(0.1, color="orange", linestyle="--", linewidth=1.5,
                label="Seuil corrélation faible (0.10)")
axes[1].set_xlabel("|Corrélation Point-Bisériale|")
axes[1].set_title("Force de corrélation avec la cible\n"
                   "(valeur absolue)", fontweight="bold")
axes[1].legend(fontsize=9)

for ax, vals in [(axes[0], pb_df["Corrélation"][::-1]),
                 (axes[1], pb_df["|Corrélation|"][::-1])]:
    for bar, val in zip(ax.patches, vals):
        ax.text(bar.get_width() + 0.002 if val >= 0 else bar.get_width() - 0.002,
                bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=8)

plt.suptitle("⚠️  Contrainte fondamentale du projet : faible corrélation variable→cible",
             fontsize=12, fontweight="bold", color="#E84855", y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "correlation_with_target.png"), dpi=120, bbox_inches="tight")
plt.show()


## 7. Analyse bivariée par classe

Boxplots et violinplots pour visualiser la séparation (ou l'absence de séparation)  
entre classes Potable / Non potable pour chaque variable.


In [63]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

df_plot = df_raw.copy()
df_plot["Classe"] = df_plot[TARGET].map(CLASS_LABELS)

for i, feat in enumerate(ALL_FEATURES):
    ax = axes[i]
    sub = df_plot[[feat, "Classe"]].dropna()

    # Violinplot
    parts = ax.violinplot(
        [sub[sub["Classe"] == "Potable"][feat].values,
         sub[sub["Classe"] == "Non potable"][feat].values],
        positions=[0, 1], showmedians=True, showextrema=False,
    )
    for pc, color in zip(parts["bodies"], PALETTE_BIN):
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    parts["cmedians"].set_color("black")
    parts["cmedians"].set_linewidth(2)

    # Boxplot superposé
    bp = ax.boxplot(
        [sub[sub["Classe"] == "Potable"][feat].values,
         sub[sub["Classe"] == "Non potable"][feat].values],
        positions=[0, 1], widths=0.15, patch_artist=True,
        boxprops=dict(alpha=0.0),
        medianprops=dict(color="black", linewidth=0),
        whiskerprops=dict(alpha=0.0),
        capprops=dict(alpha=0.0),
        flierprops=dict(marker=".", markersize=2, alpha=0.3),
    )

    # Test statistique Mann-Whitney U
    g0 = sub[sub["Classe"] == "Potable"][feat].dropna()
    g1 = sub[sub["Classe"] == "Non potable"][feat].dropna()
    if len(g0) > 5 and len(g1) > 5:
        stat, pval = stats.mannwhitneyu(g0, g1, alternative="two-sided")
        sig = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else "ns"))
        ax.set_title(f"{'✅' if feat in FEATURES else '❌'} {feat}\n"
                     f"Mann-Whitney p={pval:.3f} {sig}",
                     fontweight="bold",
                     color="#2E86AB" if feat in FEATURES else "#888888",
                     fontsize=10)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Potable", "Non potable"])
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Distribution par classe — Violin + Boxplot\n"
             "Test Mann-Whitney : *** p<0.001 | ** p<0.01 | * p<0.05 | ns = non significatif",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "bivariate_by_class.png"), dpi=120, bbox_inches="tight")
plt.show()


## 8. Contrainte de sélection des features — Justification quantifiée

Cette section formalise le compromis **performance vs coût** qui gouverne  
la sélection des 4 variables retenues.


In [111]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Groupes de features à comparer
groups = {
    "4 features retenues (budget)":    FEATURES,
    "5 features écartées (coût)":      EXCLUDED_FEATURES,
    "Toutes les 9 features":           ALL_FEATURES,
}

results = []
for label, feats in groups.items():
    sub = df_raw[feats + [TARGET]].dropna(subset=[TARGET])
    X = sub[feats]
    y = sub[TARGET]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  RobustScaler()),
        ("model",   RandomForestClassifier(n_estimators=400,
                                           class_weight="balanced",
                                           random_state=RANDOM_STATE,
                                           max_features="sqrt",
                                           min_samples_split=10,
                                           max_depth=10,
                                           n_jobs=-1)),
    ])
    scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
    results.append({
        "Groupe":         label,
        "N features":     len(feats),
        "ROC-AUC moyen":  round(scores.mean(), 4),
        "Écart-type":     round(scores.std(),  4),
        "ROC-AUC min":    round(scores.min(),  4),
        "ROC-AUC max":    round(scores.max(),  4),
    })

perf_df = pd.DataFrame(results)
display(perf_df)

# Calcul du gap de performance
auc_budget  = perf_df[perf_df["Groupe"].str.contains("retenues")]["ROC-AUC moyen"].values[0]
auc_all     = perf_df[perf_df["Groupe"].str.contains("Toutes")]["ROC-AUC moyen"].values[0]
gap         = auc_all - auc_budget

print(f"\n📊 GAP DE PERFORMANCE (9 features vs 4 features budget) : {gap:.4f} ROC-AUC")
print(f"   Ce gap quantifie le coût de la contrainte budgétaire sur la performance.")


,Groupe,N features,ROC-AUC moyen,Écart-type,ROC-AUC min,ROC-AUC max
0,4 features retenues (budget),4,0.5323,0.0132,0.5195,0.5545
1,5 features écartées (coût),5,0.5646,0.0173,0.5420,0.5946
2,Toutes les 9 features,9,0.6867,0.0209,0.6682,0.7247



📊 GAP DE PERFORMANCE (9 features vs 4 features budget) : 0.1544 ROC-AUC
   Ce gap quantifie le coût de la contrainte budgétaire sur la performance.


In [112]:
# Visualisation du gap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot ROC-AUC
colors_grp = ["#2E86AB", "#E84855", "#3BB273"]
bars = axes[0].bar(perf_df["Groupe"], perf_df["ROC-AUC moyen"],
                   yerr=perf_df["Écart-type"], color=colors_grp,
                   edgecolor="white", capsize=5, error_kw={"elinewidth":2})
axes[0].set_ylim(0.5, 0.85)
axes[0].axhline(0.5, color="gray", linestyle=":", linewidth=1, label="Baseline aléatoire")
axes[0].set_ylabel("ROC-AUC (CV 5-fold, RandomForest)")
axes[0].set_title("Impact du choix des features\nsur la performance (ROC-AUC)",
                   fontweight="bold")
axes[0].legend()
axes[0].set_xticklabels(perf_df["Groupe"], rotation=15, ha="right")
for bar, val, std in zip(bars, perf_df["ROC-AUC moyen"], perf_df["Écart-type"]):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + std + 0.005,
                 f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")

# Importance des features — modèle complet
sub_all = df_raw[ALL_FEATURES + [TARGET]].dropna(subset=[TARGET])
X_all = SimpleImputer(strategy="median").fit_transform(sub_all[ALL_FEATURES])
y_all = sub_all[TARGET].values

rf_all = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 random_state=RANDOM_STATE, n_jobs=-1)
rf_all.fit(X_all, y_all)

imp_df = pd.DataFrame({
    "Feature":    ALL_FEATURES,
    "Importance": rf_all.feature_importances_,
    "Statut":     ["✅ Retenu" if f in FEATURES else "❌ Écarté" for f in ALL_FEATURES],
}).sort_values("Importance", ascending=True)

colors_imp = ["#2E86AB" if "Retenu" in s else "#E84855" for s in imp_df["Statut"]]
axes[1].barh(imp_df["Feature"], imp_df["Importance"],
             color=colors_imp, edgecolor="white")
axes[1].set_xlabel("Importance (RandomForest, 9 features)")
axes[1].set_title("Importance des features dans le modèle complet\n"
                   "(toutes variables disponibles)", fontweight="bold")
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(facecolor="#2E86AB", label="Retenu (budget)"),
                          Patch(facecolor="#E84855", label="Écarté (coût capteur)")],
               fontsize=9)

plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "feature_importance_full.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\n⚠️  INTERPRÉTATION :")
top_excluded = imp_df[imp_df["Statut"].str.contains("Écarté")].tail(2)["Feature"].tolist()
print(f"  → Les variables les plus importantes dans le modèle complet sont souvent")
print(f"    celles que le budget ne permet pas d'acquérir : {top_excluded}")
print(f"  → La contrainte budgétaire engendre un gap de performance de ~{gap:.3f} ROC-AUC")
print(f"  → Ce gap est documenté et constitue la limite principale du système déployé.")



⚠️  INTERPRÉTATION :
  → Les variables les plus importantes dans le modèle complet sont souvent
    celles que le budget ne permet pas d'acquérir : ['Hardness', 'Sulfate']
  → La contrainte budgétaire engendre un gap de performance de ~0.154 ROC-AUC
  → Ce gap est documenté et constitue la limite principale du système déployé.


## 9. Distribution de la variable cible & déséquilibre des classes

**Convention appliquée** : `1 = Non potable` (classe dangereuse), `0 = Potable`


In [113]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribution globale
counts = df_raw[TARGET].value_counts().sort_index()
labels = [CLASS_LABELS[i] for i in counts.index]
axes[0].bar(labels, counts.values, color=PALETTE_BIN, edgecolor="white", width=0.5)
axes[0].set_title("Distribution de la variable cible", fontweight="bold")
axes[0].set_ylabel("Nombre d'observations")
for bar, val, pct in zip(axes[0].patches, counts.values, counts.values / counts.sum() * 100):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 5,
                 f"{val}\n({pct:.1f}%)", ha="center", fontsize=10)

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=labels, colors=PALETTE_BIN,
    autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2),
    textprops=dict(fontsize=11),
)
axes[1].set_title("Proportion des classes", fontweight="bold")

# Déséquilibre ratio
ratio = counts.max() / counts.min()
axes[2].text(0.5, 0.6, f"Ratio de déséquilibre\n\n{counts.max()}/{counts.min()}\n≈ {ratio:.2f}",
             ha="center", va="center", fontsize=16, fontweight="bold",
             transform=axes[2].transAxes)
axes[2].text(0.5, 0.25,
             "Stratégie appliquée :\nSMOTETomek + class_weight\n+ G-mean scorer",
             ha="center", va="center", fontsize=11, color="#E84855",
             transform=axes[2].transAxes)
axes[2].set_title("Déséquilibre & stratégie", fontweight="bold")
axes[2].axis("off")

plt.suptitle(f"Variable cible : {TARGET} — Convention : 1 = Non potable, 0 = Potable",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "class_distribution.png"), dpi=120, bbox_inches="tight")
plt.show()


## 10. Pipeline de prétraitement (`raw_data_processing`)

Exécution du pipeline complet sur les 4 features retenues.


In [114]:
import sys
sys.path.insert(0, "src")
from data_processing import raw_data_processing, preprocess_for_ml

# Appel du pipeline complet
X, y = raw_data_processing(str(RAW_DATA_PATH), return_X_y=True)

print("=" * 55)
print("  RÉSUMÉ APRÈS raw_data_processing()")
print("=" * 55)
print(f"  Shape X          : {X.shape}")
print(f"  Shape y          : {y.shape}")
print(f"  NaN résiduels    : {X.isnull().sum().sum()}")
print(f"  Distribution y   : {dict(y.value_counts().sort_index())}")
print()
display(X.describe().round(3))


2026-04-27 11:59:54 | INFO     | ============================================================
2026-04-27 11:59:54 | INFO     | DÉBUT DU PIPELINE — Water Potability
2026-04-27 11:59:54 | INFO     | ============================================================
2026-04-27 11:59:54 | INFO     | Dataset chargé : 3276 lignes × 10 colonnes
2026-04-27 11:59:54 | INFO     | Colonnes sélectionnées : ['ph', 'Solids', 'Conductivity', 'Turbidity', 'Potability']
2026-04-27 11:59:54 | INFO     | Mémoire avant : 0.13 MB  →  après : 0.05 MB  (gain : 57.4 %)
2026-04-27 11:59:54 | INFO     | Valeurs manquantes :
    n_missing  pct_missing    dtype
ph        491        14.99  float32
2026-04-27 11:59:54 | INFO     | Rapport outliers IQR (factor=1.5) :
                      Q1          Q3         IQR      lower       upper  n_outliers  pct_outliers
column                                                                                           
ph                6.0931      8.0621      1.9690     3.1396    

  RÉSUMÉ APRÈS raw_data_processing()
  Shape X          : (3276, 4)
  Shape y          : (3276,)
  NaN résiduels    : 0
  Distribution y   : {0: np.int64(1278), 1: np.int64(1998)}



,ph,Solids,Conductivity,Turbidity
count,3276.000,3276.000,3276.000,3276.000
mean,7.075,21957.112,426.130,3.967
std,1.429,8592.820,80.564,0.776
min,3.140,320.943,191.648,1.849
25%,6.278,15666.690,365.734,3.440
50%,7.035,20927.834,421.885,3.955
75%,7.870,27332.762,481.792,4.500
max,11.016,44831.869,655.879,6.091


## 11. Rééquilibrage des classes — SMOTETomek

**Stratégie** : combinaison de SMOTE (sur-échantillonnage synthétique de la classe minoritaire)  
et Tomek Links (suppression des paires ambiguës à la frontière de décision).  
Appliqué **uniquement sur le train set** pour prévenir tout data leakage.


In [115]:
from data_processing import preprocess_for_ml

X_train, X_test, y_train, y_test, scaler = preprocess_for_ml(
    X, y, test_size=0.20, random_state=42
)

print("Avant SMOTETomek :")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Classe {CLASS_LABELS[u]} ({u}) : {c} ({c/len(y_train)*100:.1f}%)")

try:
    from imblearn.combine import SMOTETomek
    from imblearn.over_sampling import SMOTE

    smote = SMOTE(k_neighbors=5, random_state=42)
    smt   = SMOTETomek(smote=smote, random_state=42)
    X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

    print("\nAprès SMOTETomek :")
    unique_r, counts_r = np.unique(y_train_res, return_counts=True)
    for u, c in zip(unique_r, counts_r):
        print(f"  Classe {CLASS_LABELS[u]} ({u}) : {c} ({c/len(y_train_res)*100:.1f}%)")
    print(f"\n  Échantillons ajoutés : {len(y_train_res) - len(y_train)}")

    # Visualisation avant/après
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, X_d, y_d, title in [
        (axes[0], X_train,     y_train,     "Avant SMOTETomek (train set)"),
        (axes[1], X_train_res, y_train_res, "Après SMOTETomek (train set)"),
    ]:
        for cls, color in [(0, PALETTE_BIN[0]), (1, PALETTE_BIN[1])]:
            mask = y_d == cls
            ax.scatter(X_d[mask, 0], X_d[mask, 1], c=color, alpha=0.4,
                       s=15, label=CLASS_LABELS[cls])
        ax.set_xlabel("ph (standardisé)")
        ax.set_ylabel("Solids (standardisé)")
        ax.set_title(title, fontweight="bold")
        ax.legend()

    plt.tight_layout()
    plt.savefig(str(OUTPUTS_PATH / "smotetomek.png"), dpi=120, bbox_inches="tight")
    plt.show()

except ImportError:
    print("\n⚠️  imbalanced-learn non installé.")
    print("   pip install imbalanced-learn")


2026-04-27 11:59:54 | INFO     | Split train/test : 2620 / 656  (stratifie, test_size=20%)
2026-04-27 11:59:54 | INFO     |   Train -- distribution Potability : {1: 0.61, 0: 0.39}
2026-04-27 11:59:54 | INFO     |   Test -- distribution Potability : {1: 0.61, 0: 0.39}
2026-04-27 11:59:54 | INFO     | RobustScaler fitte sur le train set. Centre (mediane) : [7.0350000e+00 2.1051312e+04 4.2144100e+02 3.9430000e+00] | Echelle (IQR) : [1.593000e+00 1.155755e+04 1.144790e+02 1.063000e+00]


Avant SMOTETomek :
  Classe Potable (0) : 1022 (39.0%)
  Classe Non potable (1) : 1598 (61.0%)

Après SMOTETomek :
  Classe Potable (0) : 1406 (50.0%)
  Classe Non potable (1) : 1406 (50.0%)

  Échantillons ajoutés : 192


## 12. Résumé & Points clés avant modélisation

| Aspect | Valeur / Choix |
|---|---|
| **Observations** | 3 276 |
| **Features retenues** | 4 (ph, Solids, Conductivity, Turbidity) |
| **Cible** | Potability — `1 = Non potable`, `0 = Potable` |
| **Déséquilibre** | ~61% Non potable / ~39% Potable |
| **Stratégie déséquilibre** | SMOTETomek + class_weight="balanced" |
| **Standardisation** | RobustScaler (robuste aux outliers) |
| **Scorer GridSearch** | G-mean = √(recall₁ × recall₀) |
| **Scorer classement** | Score composite pondéré (5 métriques) |
| **Validation croisée** | StratifiedKFold (10 folds) |

### ⚠️ Limite principale du projet

La corrélation entre les 4 features retenues et la cible est **faible**  
(max ~0.08–0.12 en valeur absolue). Les variables les plus prédictives  
(Sulfate, Organic Carbon, Trihalomethanes) nécessitent des capteurs  
coûteux hors budget. Ce système doit être interprété comme un **outil  
de triage à faible coût**, et non comme un substitut à une analyse  
physico-chimique complète en laboratoire.

---

### Prochaines étapes

```bash
# 1. Optimisation des hyperparamètres
python src/tuning.py

# 2. Entraînement, évaluation et sauvegarde des top-3 modèles
python src/train_model.py
```


---

## 13. Analyse par Clustering — Trois expériences

> **Objectif** : Explorer la structure non supervisée des données pour  
> (1) quantifier le chevauchement des classes,  
> (2) enrichir les features par GMM,  
> (3) tester l'Isolation Forest comme classifieur alternatif.

```
Expérience 1 : KMeans(k=2)         → ARI score — mesure le chevauchement
Expérience 2 : GMM feature eng.    → X enrichi → classifieur supervisé
Expérience 3 : Isolation Forest    → détection d'anomalies standalone
```


In [116]:
# Imports spécifiques au clustering
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    adjusted_rand_score, silhouette_score,
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, confusion_matrix,
)
import warnings
warnings.filterwarnings("ignore")

# Préparer X et y propres (après raw_data_processing)
import sys; sys.path.insert(0, "src")
from data_processing import raw_data_processing, preprocess_for_ml

X, y = raw_data_processing(str(RAW_DATA_PATH), return_X_y=True)
X_train, X_test, y_train, y_test, scaler = preprocess_for_ml(
    X, y, test_size=0.20, random_state=42
)

print(f"X_train : {X_train.shape} | X_test : {X_test.shape}")
print(f"Distribution y_train : {dict(zip(*np.unique(y_train, return_counts=True)))}")


2026-04-27 11:59:56 | INFO     | ============================================================
2026-04-27 11:59:56 | INFO     | DÉBUT DU PIPELINE — Water Potability
2026-04-27 11:59:56 | INFO     | ============================================================
2026-04-27 11:59:56 | INFO     | Dataset chargé : 3276 lignes × 10 colonnes
2026-04-27 11:59:56 | INFO     | Colonnes sélectionnées : ['ph', 'Solids', 'Conductivity', 'Turbidity', 'Potability']
2026-04-27 11:59:56 | INFO     | Mémoire avant : 0.13 MB  →  après : 0.05 MB  (gain : 57.4 %)
2026-04-27 11:59:56 | INFO     | Valeurs manquantes :
    n_missing  pct_missing    dtype
ph        491        14.99  float32
2026-04-27 11:59:56 | INFO     | Rapport outliers IQR (factor=1.5) :
                      Q1          Q3         IQR      lower       upper  n_outliers  pct_outliers
column                                                                                           
ph                6.0931      8.0621      1.9690     3.1396    

X_train : (2620, 4) | X_test : (656, 4)
Distribution y_train : {np.int8(0): np.int64(1022), np.int8(1): np.int64(1598)}


### Expérience 1 — KMeans(k=2) : les données forment-elles naturellement 2 groupes ?

**Hypothèse testée** : si les 4 features structurent bien les deux classes,  
KMeans devrait retrouver une partition proche de Potable / Non potable.  
L'**Adjusted Rand Index (ARI)** mesure cette concordance :
- ARI = 1.0 → clustering parfait = classes retrouvées
- ARI = 0.0 → clustering aléatoire = classes non séparables dans cet espace
- ARI < 0.1 → chevauchement confirmé


In [117]:
# ── KMeans k=2 ───────────────────────────────────────────────────────────
kmeans2 = KMeans(n_clusters=2, random_state=42, n_init=20)
clusters2 = kmeans2.fit_predict(X_train)

ari2 = adjusted_rand_score(y_train, clusters2)
sil2 = silhouette_score(X_train, clusters2)

print("=" * 55)
print("  EXPÉRIENCE 1 — KMeans(k=2)")
print("=" * 55)
print(f"  Adjusted Rand Index (ARI) : {ari2:.4f}")
print(f"  Silhouette Score          : {sil2:.4f}")
print()

if ari2 < 0.05:
    verdict = "⚠️  CHEVAUCHEMENT FORT — KMeans ne retrouve pas les classes."
elif ari2 < 0.15:
    verdict = "⚠️  STRUCTURE FAIBLE — légère tendance, non exploitable seule."
else:
    verdict = "✅ STRUCTURE DÉTECTÉE — les features discriminent partiellement."
print(f"  Verdict : {verdict}")

# Correspondance clusters → classes
print("\n  Correspondance clusters → classes réelles :")
for c in [0, 1]:
    mask = clusters2 == c
    pct_np = y_train[mask].mean() * 100
    print(f"    Cluster {c} ({mask.sum()} pts) : "
          f"{pct_np:.1f}% Non potable / {100-pct_np:.1f}% Potable")


  EXPÉRIENCE 1 — KMeans(k=2)
  Adjusted Rand Index (ARI) : -0.0004
  Silhouette Score          : 0.1776

  Verdict : ⚠️  CHEVAUCHEMENT FORT — KMeans ne retrouve pas les classes.

  Correspondance clusters → classes réelles :
    Cluster 0 (1383 pts) : 59.9% Non potable / 40.1% Potable
    Cluster 1 (1237 pts) : 62.2% Non potable / 37.8% Potable


In [118]:
# ── Courbe d'inertie (Elbow) pour choisir k optimal ─────────────────────
inertias, aris, sils = [], [], []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_train)
    inertias.append(km.inertia_)
    aris.append(adjusted_rand_score(y_train, labels))
    sils.append(silhouette_score(X_train, labels))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(k_range, inertias, "o-", color="#2E86AB", lw=2)
axes[0].set_xlabel("Nombre de clusters k")
axes[0].set_ylabel("Inertie")
axes[0].set_title("Méthode du coude (Elbow)", fontweight="bold")
axes[0].axvline(2, color="red", linestyle="--", alpha=0.5, label="k=2 (classes réelles)")
axes[0].legend()

axes[1].plot(k_range, sils, "s-", color="#3BB273", lw=2)
axes[1].set_xlabel("Nombre de clusters k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score par k\n(plus élevé = meilleure cohésion)", fontweight="bold")

axes[2].plot(k_range, aris, "^-", color="#E84855", lw=2)
axes[2].set_xlabel("Nombre de clusters k")
axes[2].set_ylabel("Adjusted Rand Index")
axes[2].set_title("ARI vs classes réelles\n(plus élevé = mieux aligné)", fontweight="bold")
axes[2].axhline(0, color="gray", linestyle=":", linewidth=1)

plt.suptitle("KMeans — Recherche du k optimal", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "kmeans_elbow.png"), dpi=120, bbox_inches="tight")
plt.show()


In [119]:
# ── Visualisation 2D par PCA ──────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train)
var_exp = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vue classes réelles
for cls, color, label in [(0, "#2E86AB", "Potable"), (1, "#E84855", "Non potable")]:
    mask = y_train == cls
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, alpha=0.4, s=12, label=label)
axes[0].set_title(f"Classes réelles (PCA)\n"
                   f"PC1={var_exp[0]:.1%} | PC2={var_exp[1]:.1%}",
                   fontweight="bold")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].legend()

# Vue clusters KMeans
cmap_km = plt.cm.get_cmap("Set1", 2)
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                      c=clusters2, cmap=cmap_km, alpha=0.4, s=12)
# Centroïdes projetés
centroids_pca = pca.transform(kmeans2.cluster_centers_)
axes[1].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                c="black", s=120, marker="X", zorder=5, label="Centroïdes")
axes[1].set_title(f"Clusters KMeans(k=2) (PCA)\nARI = {ari2:.4f}",
                   fontweight="bold")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].legend()

plt.suptitle("Comparaison : classes réelles vs clusters KMeans\n"
             "Un fort chevauchement confirme la difficulté du problème",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "kmeans_pca.png"), dpi=120, bbox_inches="tight")
plt.show()

print(f"\n📊 Variance expliquée par PC1+PC2 : {sum(var_exp):.1%}")
print(f"   ARI KMeans(k=2) vs classes réelles : {ari2:.4f}")
print(f"   → {'Chevauchement confirmé' if ari2 < 0.1 else 'Structure partielle détectée'}")



📊 Variance expliquée par PC1+PC2 : 57.7%
   ARI KMeans(k=2) vs classes réelles : -0.0004
   → Chevauchement confirmé


### Expérience 2 — GMM comme feature engineering

**Idée** : un Gaussian Mixture Model capture des structures ellipsoïdales  
dans les données. Les probabilités d'appartenance à chaque composante  
deviennent de **nouvelles features** — potentiellement plus discriminantes  
que les mesures brutes.

```
X (4 features) ──→ GMM (k composantes) ──→ P(composante_i | x)  ← nouvelles features
                                              ↓
                              X enrichi (4 + k features) → RF / XGB
```

**Important** : le GMM est fitté **uniquement sur X_train** pour éviter  
tout data leakage. X_test est transformé avec le GMM déjà fitté.


In [120]:
# ── Recherche du nombre optimal de composantes GMM ──────────────────────
bic_scores, aic_scores = [], []
k_range_gmm = range(2, 9)

for k in k_range_gmm:
    gmm = GaussianMixture(n_components=k, covariance_type="full",
                          random_state=42, n_init=5)
    gmm.fit(X_train)
    bic_scores.append(gmm.bic(X_train))
    aic_scores.append(gmm.aic(X_train))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_range_gmm, bic_scores, "o-", label="BIC", color="#2E86AB", lw=2)
ax.plot(k_range_gmm, aic_scores, "s-", label="AIC", color="#E84855", lw=2)
ax.set_xlabel("Nombre de composantes k")
ax.set_ylabel("Score (plus bas = meilleur)")
ax.set_title("Sélection du nombre de composantes GMM\n"
             "BIC / AIC — choisir le coude ou le minimum", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "gmm_bic_aic.png"), dpi=120, bbox_inches="tight")
plt.show()

best_k = k_range_gmm[np.argmin(bic_scores)]
print(f"Nombre de composantes optimal (BIC) : k = {best_k}")


Nombre de composantes optimal (BIC) : k = 4


In [121]:
# ── Comparaison performance : features brutes vs features enrichies GMM ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results_gmm = []

for k in [2, 3, 4, best_k]:
    # Fit GMM sur train uniquement
    gmm = GaussianMixture(n_components=k, covariance_type="full",
                          random_state=42, n_init=5)
    gmm.fit(X_train)

    # Nouvelles features : probabilités d'appartenance + distance log-vraisemblance
    gmm_proba_train = gmm.predict_proba(X_train)   # (n, k)
    gmm_proba_test  = gmm.predict_proba(X_test)

    # Enrichissement
    X_train_enr = np.hstack([X_train, gmm_proba_train])
    X_test_enr  = np.hstack([X_test,  gmm_proba_test])

    # Classifieur RF sur features enrichies
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                 random_state=42, n_jobs=-1)
    rf.fit(X_train_enr, y_train)
    y_proba_enr = rf.predict_proba(X_test_enr)[:, 1]
    auc_enr = roc_auc_score(y_test, y_proba_enr)

    results_gmm.append({
        "k composantes GMM": k,
        "N features total":  4 + k,
        "ROC-AUC (enrichi)": round(auc_enr, 4),
    })

# Baseline : RF sur 4 features brutes
rf_base = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                  random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
auc_base = roc_auc_score(y_test, rf_base.predict_proba(X_test)[:, 1])

df_gmm = pd.DataFrame(results_gmm)
df_gmm["Gain vs baseline"] = (df_gmm["ROC-AUC (enrichi)"] - auc_base).round(4)
df_gmm["Gain vs baseline"] = df_gmm["Gain vs baseline"].apply(
    lambda x: f"+{x:.4f} ✅" if x > 0.01 else (f"+{x:.4f} ⚠️" if x > 0 else f"{x:.4f} ❌")
)

print(f"  Baseline RF (4 features brutes) : ROC-AUC = {auc_base:.4f}")
print()
display(df_gmm)


  Baseline RF (4 features brutes) : ROC-AUC = 0.6672



,k composantes GMM,N features total,ROC-AUC (enrichi),Gain vs baseline
0,2,6,0.6528,-0.0144 ❌
1,3,7,0.6660,-0.0012 ❌
2,4,8,0.6507,-0.0165 ❌
3,4,8,0.6507,-0.0165 ❌


In [122]:
# ── Visualisation du gain GMM ─────────────────────────────────────────────
aucs_enr = [r["ROC-AUC (enrichi)"] for r in results_gmm]
k_vals   = [r["k composantes GMM"] for r in results_gmm]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar([f"GMM k={k}" for k in k_vals], aucs_enr,
              color=["#3BB273" if a > auc_base + 0.01 else
                     "#F18F01" if a > auc_base else "#E84855"
                     for a in aucs_enr],
              edgecolor="white", width=0.5)
ax.axhline(auc_base, color="#2E86AB", linestyle="--", lw=2,
           label=f"Baseline 4 features (AUC={auc_base:.4f})")
ax.set_ylabel("ROC-AUC (Test set)")
ax.set_title("Impact du feature engineering GMM\nsur la performance du RandomForest",
             fontweight="bold")
ax.set_ylim(max(0.5, auc_base - 0.05), min(1.0, max(aucs_enr) + 0.05))
ax.legend()

for bar, val in zip(bars, aucs_enr):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002,
            f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "gmm_feature_engineering.png"), dpi=120, bbox_inches="tight")
plt.show()

best_k_gmm = k_vals[np.argmax(aucs_enr)]
best_auc_gmm = max(aucs_enr)
gain = best_auc_gmm - auc_base
print(f"\n📊 RÉSULTAT EXPÉRIENCE 2")
print(f"  Meilleure config  : GMM k={best_k_gmm}  →  AUC = {best_auc_gmm:.4f}")
print(f"  Gain vs baseline  : {gain:+.4f}")
if gain > 0.02:
    print("  ✅ Gain significatif — intégrer GMM dans data_processing.py")
elif gain > 0:
    print("  ⚠️  Gain marginal — à valider sur d'autres métriques")
else:
    print("  ❌ Pas de gain — GMM n'apporte pas de structure exploitable")



📊 RÉSULTAT EXPÉRIENCE 2
  Meilleure config  : GMM k=3  →  AUC = 0.6660
  Gain vs baseline  : -0.0012
  ❌ Pas de gain — GMM n'apporte pas de structure exploitable


### Expérience 3 — Isolation Forest : détection d'anomalies

**Changement de paradigme** : au lieu de classifier directement,  
on suppose que l'eau **non potable est une anomalie** par rapport  
à la distribution normale de l'eau potable.

```
Entraînement : uniquement sur les eaux potables (classe 0)
Inférence    : score d'anomalie → eau non potable si score élevé
```

**Avantage** : pas besoin d'étiquettes pour la classe Non potable  
pendant l'entraînement — utile si les données non potables sont rares  
ou peu fiables dans un futur déploiement terrain.


In [123]:
# ── Isolation Forest — entraîné sur eaux potables uniquement ─────────────
# Séparer les classes dans le train set
X_potable     = X_train[y_train == 0]
X_non_potable = X_train[y_train == 1]

print(f"Train set — Potable : {len(X_potable)} | Non potable : {len(X_non_potable)}")

# contamination = proportion attendue d'anomalies dans les données de test
contamination = min(float((y_test == 1).mean()), 0.499)  # plafonne a 0.499 (limite sklearn)
print(f"Contamination estimée (test set) : {contamination:.3f}")

iso = IsolationForest(
    n_estimators=300,
    contamination=float(contamination),
    max_features=1.0,
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_potable)   # UNIQUEMENT sur les eaux potables

# Prédictions : -1 = anomalie (non potable), 1 = normal (potable)
y_pred_iso_raw = iso.predict(X_test)
y_pred_iso = (y_pred_iso_raw == -1).astype(int)   # -1 → 1 (Non potable)

# Scores d'anomalie (plus négatif = plus anormal)
scores_iso = -iso.score_samples(X_test)   # inversé : plus élevé = plus anormal

print("\n" + "=" * 55)
print("  EXPÉRIENCE 3 — Isolation Forest")
print("=" * 55)
print(classification_report(y_test, y_pred_iso,
                              target_names=["Potable", "Non potable"]))

auc_iso = roc_auc_score(y_test, scores_iso)
print(f"  ROC-AUC (score d'anomalie) : {auc_iso:.4f}")


Train set — Potable : 1022 | Non potable : 1598
Contamination estimée (test set) : 0.499

  EXPÉRIENCE 3 — Isolation Forest
              precision    recall  f1-score   support

     Potable       0.40      0.50      0.44       256
 Non potable       0.62      0.52      0.56       400

    accuracy                           0.51       656
   macro avg       0.51      0.51      0.50       656
weighted avg       0.53      0.51      0.52       656

  ROC-AUC (score d'anomalie) : 0.5036


In [124]:
# ── Comparaison : Isolation Forest vs classifieurs supervisés ────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Matrice de confusion Isolation Forest
cm_iso = confusion_matrix(y_test, y_pred_iso)
ConfusionMatrixDisplay(cm_iso, display_labels=["Potable", "Non potable"]).plot(
    ax=axes[0], colorbar=False, cmap="Oranges"
)
axes[0].set_title(f"Isolation Forest\n(ROC-AUC = {auc_iso:.4f})",
                   fontweight="bold")
axes[0].set_xlabel("Prédit"); axes[0].set_ylabel("Réel")

# 2. Distribution des scores d'anomalie par classe
for cls, color, label in [(0, "#2E86AB", "Potable"), (1, "#E84855", "Non potable")]:
    mask = y_test == cls
    axes[1].hist(scores_iso[mask], bins=40, alpha=0.55,
                  color=color, label=label, density=True)
axes[1].axvline(np.percentile(scores_iso, (1 - contamination) * 100),
                color="black", linestyle="--", lw=1.5, label="Seuil décision")
axes[1].set_xlabel("Score d'anomalie (plus élevé = plus suspect)")
axes[1].set_ylabel("Densité")
axes[1].set_title("Distribution des scores\npar classe réelle", fontweight="bold")
axes[1].legend()

# 3. Comparaison ROC-AUC finale
models_compare = {
    "RF\n(supervisé)":    roc_auc_score(y_test, rf_base.predict_proba(X_test)[:, 1]),
    f"RF+GMM\n(k={best_k_gmm})": best_auc_gmm,
    "Isolation\nForest":  auc_iso,
}
colors_cmp = ["#2E86AB", "#3BB273", "#F18F01"]
bars = axes[2].bar(models_compare.keys(), models_compare.values(),
                    color=colors_cmp, edgecolor="white", width=0.5)
axes[2].set_ylabel("ROC-AUC (Test set)")
axes[2].set_title("Comparaison finale\nsupervisé vs non supervisé",
                   fontweight="bold")
axes[2].set_ylim(0.5, 0.85)
axes[2].axhline(0.5, color="gray", linestyle=":", lw=1, label="Baseline aléatoire")
axes[2].legend(fontsize=9)
for bar, val in zip(bars, models_compare.values()):
    axes[2].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                 f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")

plt.suptitle("Expérience 3 — Isolation Forest vs approches supervisées",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(str(OUTPUTS_PATH / "isolation_forest.png"), dpi=120, bbox_inches="tight")
plt.show()


In [125]:
# ── Tuning du seuil de score pour Isolation Forest ───────────────────────
from sklearn.metrics import fbeta_score

thresholds_iso = np.percentile(scores_iso, np.arange(10, 90, 1))
best_thresh_iso, best_fbeta_iso = None, -1

for t in thresholds_iso:
    y_pred_t = (scores_iso >= t).astype(int)
    fb = fbeta_score(y_test, y_pred_t, beta=2, zero_division=0)
    if fb > best_fbeta_iso:
        best_fbeta_iso = fb
        best_thresh_iso = t

y_pred_iso_tuned = (scores_iso >= best_thresh_iso).astype(int)

print("=" * 55)
print(f"  Seuil optimal Isolation Forest : {best_thresh_iso:.4f}")
print(f"  Fbeta(β=2) optimal             : {best_fbeta_iso:.4f}")
print()
print(classification_report(y_test, y_pred_iso_tuned,
                              target_names=["Potable", "Non potable"]))


  Seuil optimal Isolation Forest : 0.4030
  Fbeta(β=2) optimal             : 0.8174

              precision    recall  f1-score   support

     Potable       0.36      0.09      0.15       256
 Non potable       0.61      0.90      0.72       400

    accuracy                           0.58       656
   macro avg       0.49      0.49      0.44       656
weighted avg       0.51      0.58      0.50       656



### Synthèse des trois expériences

| Expérience | Résultat attendu si chevauchement fort | Ce que ça implique |
|---|---|---|
| **KMeans ARI** | ARI ≈ 0 | Les 4 features ne séparent pas les classes dans l'espace euclidien |
| **GMM features** | Gain < 0.02 AUC | Pas de structure latente exploitable par des composantes gaussiennes |
| **Isolation Forest** | AUC < RF supervisé | L'eau non potable n'est pas une anomalie simple — elle se confond avec le potable |

**Conclusion générale**  
Ces trois expériences servent à **confirmer quantitativement** la contrainte  
identifiée dans la section 8 : avec les 4 features retenues, la frontière  
de décision est intrinsèquement floue. Les approches non supervisées ne  
compensent pas le manque d'information — elles le confirment.

> Le seul levier réel reste l'acquisition de données supplémentaires  
> (variables hors budget) ou l'enrichissement du signal par la caméra  
> (turbidité par vision).
